# Tools: Giving LLM Agents the Ability to Act

An LLM on its own can only generate text. **Tools** let an agent look things up, run code, and call external services — the model decides *when* and *how* to call a tool, and your code executes it.

This notebook walks through five tool patterns, mostly using [`smolagents`](https://github.com/huggingface/smolagents) (with `LiteLLMModel` as a thin, provider-agnostic wrapper around `gpt-4o-mini`), plus one example using the native OpenAI Agents SDK:

1. **Custom Database Tool** — a `@tool`-decorated function that looks up order status.
2. **Python Interpreter Tool** — letting the agent execute Python to solve a computation.
3. **Search + Calculator** — combining a web search tool with the agent's own reasoning/math.
4. **Stock Price Tool** — a `function_tool` wired into the OpenAI Agents SDK, backed by `yfinance`.
5. **Web Browser Tool** — an agent that writes its own scraping code against a live webpage.

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q smolagents litellm duckduckgo-search yfinance openai-agents python-dotenv beautifulsoup4 httpx

---
## Step 2: Set Up Environment Variables
All five demos call `gpt-4o-mini` under the hood, so we only need an OpenAI API key.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

---
## Tool 1: Custom Database Lookup
We define a plain Python function, decorate it with `@tool`, and hand it to a `CodeAgent`. The docstring's `Args` section is what the model reads to know how to call it — so it doubles as the tool's API contract.

> The original version of this demo pointed at a local Ollama model (`ollama/gemma3:4b` via `api_base="http://localhost:11434"`). Here we swap that for the hosted `gpt-4o-mini` so the notebook runs end-to-end in Colab with no local server.

In [ ]:
from smolagents import CodeAgent, LiteLLMModel, tool

model = LiteLLMModel(model_id="gpt-4o-mini")

@tool
def get_order_status(order_id: str) -> str:
    """
    Retrieves the real-time status and estimated delivery for a customer order.

    Args:
        order_id: The unique alphanumeric order tracking ID (e.g., 'ORD-123').
    """
    # Simulated Database Lookup
    mock_db = {
        "ORD-101": {"status": "In Transit", "location": "Bengaluru Hub", "eta": "2 Hours"},
        "ORD-202": {"status": "Delivered", "location": "Mumbai", "eta": "N/A"},
        "ORD-303": {"status": "Processing", "location": "Warehouse", "eta": "2 Days"}
    }

    order = mock_db.get(order_id.upper())
    if order:
        return f"Order {order_id} is currently '{order['status']}' at {order['location']}. Estimated arrival: {order['eta']}."
    return f"Order ID {order_id} not found. Please verify the ID and try again."

agent = CodeAgent(
    tools=[get_order_status],
    model=model,
    add_base_tools=False  # We only want it to use our specific support tool
)

customer_query = "Hey, I've been waiting for my package ORD-101. Can you tell me where it is and when it will get here?"
agent.run(customer_query)

---
## Tool 2: Python Interpreter
`PythonInterpreterTool` gives the agent a sandboxed Python runtime it can call to compute exact answers instead of guessing numerically.

In [ ]:
from smolagents import ToolCallingAgent, PythonInterpreterTool, LiteLLMModel

model = LiteLLMModel(model_id="gpt-4o-mini")

# Create a traditional JSON-based tool-calling agent with access to Python code execution
agent = ToolCallingAgent(
    tools=[PythonInterpreterTool()],
    model=model
)

result = agent.run("What is the 50th Fibonacci number? Compute it using Python.")
print(result)

---
## Tool 3: Search + Calculator
`DuckDuckGoSearchTool` gives the agent live web access. `CodeAgent` can then combine that retrieved fact with its own arithmetic — no separate calculator tool is needed since the agent writes and runs Python directly.

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel

model = LiteLLMModel(model_id="gpt-4o-mini")

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model
)

result = agent.run(
    "Search for the distance between Earth and Mars today, "
    "and calculate how long light takes to travel that distance in minutes."
)
print(result)

---
## Tool 4: Stock Price Lookup (OpenAI Agents SDK)
This example switches frameworks: instead of `smolagents`, we use OpenAI's native **Agents SDK**, which packages instructions, models, and tools directly into an `Agent` object. `@function_tool` turns any typed Python function into a callable tool, and `Runner` drives the LLM → tool call → result → LLM loop automatically.

In [ ]:
import yfinance as yf
from agents import Agent, Runner, function_tool

@function_tool
def get_stock_price(ticker: str) -> str:
    info = yf.Ticker(ticker).fast_info
    return f"The price of {ticker} is {info.last_price:.2f} {info.currency}."

finance_agent = Agent(
    name="Finance Agent",
    instructions="You are a financial assistant. Use tools to look up prices.",
    tools=[get_stock_price],
    model="gpt-4o",
)

result = await Runner.run(finance_agent, "What is the stock price of AAPL?")
print(result.final_output)

---
## Tool 5: Web Browser
Earlier iterations of this demo let the agent write its own scraping and DOM-selection code from scratch. That turned out to be unreliable: the model guessed CSS selectors from its training memory (e.g. Hacker News's old `storylink` class, retired years ago) instead of the site's current markup, and the scrape failed.

Here we fix that by making the scraping logic a **deterministic tool** — `scrape_hacker_news` — instead of freeform agent-written code. The agent's job shrinks to calling the tool and formatting the result; the parsing itself no longer depends on the model guessing correctly.

In [ ]:
from smolagents import CodeAgent, LiteLLMModel, tool
import httpx
import json
from bs4 import BeautifulSoup

model = LiteLLMModel(model_id="gpt-4o-mini")

@tool
def scrape_hacker_news(top_n: int) -> str:
    """
    Fetches the Hacker News front page and returns the top stories' titles and point counts as JSON.

    Args:
        top_n: Number of top stories to return.
    """
    response = httpx.get("https://news.ycombinator.com/", timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    stories = []
    for row in soup.find_all("tr", class_="athing")[:top_n]:
        title_el = row.select_one(".titleline > a")
        title = title_el.get_text() if title_el else "Unknown"

        subtext_row = row.find_next_sibling("tr")
        score_el = subtext_row.find(class_="score") if subtext_row else None
        points = score_el.get_text().split()[0] if score_el else "0"

        stories.append({"title": title, "points": points})

    return json.dumps(stories)

agent = CodeAgent(
    tools=[scrape_hacker_news],
    model=model,
    add_base_tools=False,
)

result = agent.run("Get the top 5 Hacker News stories with their titles and point counts.")
print(result)